In [1]:
%pip install -q langchain_core langchain-google-genai pydantic python-dotenv

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os


In [ ]:
load_dotenv()
APIKEY = os.environ.get("GEMINI_API_KEY")

In [4]:
class Actor(BaseModel):
    nome: str = Field(description="O nome dos atores e atrizes")
    personagem: str = Field(description="O nome do personagem que interpretaram.")


class MovieSummary(BaseModel):
    title: str = Field(description="O título do filme.")
    release_year: int = Field(description="O ano em que o filme foi lançado.")
    genres: List[str] = Field(description="Uma lista de gêneros para o filme.")
    main_cast: List[Actor] = Field(description="Uma lista dos 3 principais atores.")

In [7]:
parser = PydanticOutputParser(pydantic_object=MovieSummary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce é um assistente de cinema ajudante. Sempre envolva sua resposta final estritamente em JSON correspondente ao esquema solicitado.\n{format_instructions}"),
    ("user", "Extraia as informações sobre o filme: {movie}")
]).partial(format_instructions=parser.get_format_instructions())

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",
                   api_key = APIKEY,
                   temperature=0)

chain = prompt | model | parser

In [9]:
description_movie = (
    "O filme 'Inception' foi lançado em 2010. É um thriller de ficção científica dirigido por Christopher Nolan." 
    "O elenco principal inclui Leonardo DiCaprio como Dom Cobb, Joseph Gordon-Levitt como Arthur e Ellen Page como Ariadne."
    " O filme explora o conceito de sonhos dentro de sonhos e a manipulação da mente humana."
)

In [10]:
chain.invoke({"movie":description_movie })

MovieSummary(title='Inception', release_year=2010, genres=['Sci-Fi', 'Thriller'], main_cast=[Actor(nome='Leonardo DiCaprio', personagem='Dom Cobb'), Actor(nome='Joseph Gordon-Levitt', personagem='Arthur'), Actor(nome='Ellen Page', personagem='Ariadne')])